# Persona pilot: Gemma 3 1B IT + LoRA on 60 examples

Model selection is closed: `google/gemma-3-1b-it`. This notebook is the small pilot before any larger dataset.

- **Data:** 60 examples, 10 scenarios × 6 personas (Vijay of *Deewar*, Don, Shahenshah, Anthony Gonsalves, Vijay Dinanath Chauhan of *Agneepath*, Jai of *Sholay*). One Roman-script Hinglish line per example.
- **Split by scenario, not by example:** 8 scenarios train (48 examples); 2 scenarios are held out (12 examples, with reference lines). Six new scenarios, not in the dataset at all, are added for evaluation (no reference lines).
- **Training:** LoRA on the attention and MLP projections; the loss is on the answer only.
- **Evaluation:** the same inputs through untouched Gemma (adapter switched off) and through Gemma + LoRA. Two separate checks:
  - **content quality:** script, clean stop, relevance, naturalness (human rubric);
  - **persona differentiation:** do the six personas give meaningfully different lines for the same scenario (automatic overlap numbers plus a blind persona-matching sheet).

Either Devanagari or Roman-script Hinglish is acceptable for now, if it is natural and consistent; the training lines are Roman script, so Roman script is the expected style.

### Run 2 (this version)

Run 1 (commit `e5d9557`) used the bare tag `[Persona: Don]`. LoRA learned the format (stopped 47/48, Roman script 48/48, no preamble) but not the content: lines were mostly ungrammatical, 16/48 repeated a sentence, and the personas overlapped 0.303 (the dataset's own lines: 0.087). Held-out loss was lowest at epoch 2 and rose after. Three changes, same data and split:

1. **Persona descriptions in the input:** two lines of broad traits per persona, in training and evaluation alike (Cell 2).
2. **Lighter training:** 3 epochs at learning rate 1e-4 (was 6 at 2e-4); the best held-out epoch is still kept (Cell 4).
3. **No repeated 3-grams at generation** (`no_repeat_ngram_size=3`), for both variants; otherwise still greedy (Cell 5).

**Success bar, fixed before the run:**

| Check | Run 1 | Pass |
|---|---|---|
| Mean overlap between personas | 0.303 | ≤ 0.15 |
| Outputs repeating a sentence | 16/48 | ≤ 4/48 |
| Readable, on-topic lines (human) | almost none | most of the 48 |
| Blind persona matching (Cell 8) | — | at least half right (chance is 1 in 6) |

In [18]:
# Cell 0 — Hugging Face token: paste it here before running (needed for the gated Gemma 3)
# Clear it again before saving or sharing the notebook, so the token never ends up in a file.

HF_TOKEN = ""

## Cell 1 — Setup

Same libraries and settings as the Hinglish comparison: fp32 (a T4 has no fast bfloat16, and Gemma 3 is unstable in fp16), greedy decoding, 80 new tokens at most.

In [19]:
# Cell 1 — Setup: packages, settings, versions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import json, random, re, time, urllib.request
from itertools import combinations

import peft
import torch
import transformers
from huggingface_hub import HfApi, login

GEMMA = "google/gemma-3-1b-it"
DATA_URL = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
            "hindi-decoder-compare/project4-hindi-decoder/data/persona_pilot_60.json")
MAX_NEW_TOKENS = 80
SEED = 42
DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

HF_TOKEN = HF_TOKEN.strip() or None
if HF_TOKEN:
    login(HF_TOKEN)
REV = HfApi().model_info(GEMMA, token=HF_TOKEN).sha
VERSIONS = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "cpu",
    "dtype": str(DTYPE),
    "gemma revision": REV,
}
print(VERSIONS)

{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'peft': '0.21.2', 'device': 'Tesla T4', 'dtype': 'torch.float32', 'gemma revision': 'dcc83ea841ab6100d6b47a070329e1ba4cf78752'}


## Cell 2 — Data and the split

The 60 examples come from the repository. The two held-out scenarios are chosen before training and never seen by it: one confrontation (*being provoked*), one everyday moment (*first day in a new city*). The six new scenarios use the same input wording as the dataset, so only the scenario changes.

Every input is rebuilt as the dataset's input with one added line, the persona's broad traits (no film dialogue):

```
[Persona: Don] Traits: <two lines>
Scenario: ... Generate one original Roman-script Hinglish dialogue line ... Do not quote existing films.
```

In [20]:
# Cell 2 — Load the 60 examples, split by scenario, add six new scenarios

DATA = json.load(urllib.request.urlopen(DATA_URL))
PARSE = re.compile(r"\[Persona: (\w+)\] Scenario: (.+?) Generate one")
for ex in DATA:
    ex["persona"], ex["scenario"] = PARSE.match(ex["input"]).groups()

PERSONAS = list(dict.fromkeys(ex["persona"] for ex in DATA))
SCENARIOS = list(dict.fromkeys(ex["scenario"] for ex in DATA))
TAIL = DATA[0]["input"].split(SCENARIOS[0])[1]          # " Generate one original Roman-script ... films."

HELD_OUT = [
    "Koi tumhe gusse mein provoke kar raha hai.",
    "Aaj tumhara pehla din hai ek naye shehar mein.",
]
NEW = [
    "Boss ne Friday shaam ko achaanak weekend ka kaam de diya.",
    "Tumhara purana dost saalon baad airport par mil gaya.",
    "Presentation se paanch minute pehle office ka WiFi band ho gaya.",
    "Baarish mein gaadi beech sadak par band ho gayi.",
    "Team ka sabse senior saathi aaj retire ho raha hai.",
    "Kisi ne tumse kaha ki tum yeh kaam kabhi nahi kar paoge.",
]
assert all(s in SCENARIOS for s in HELD_OUT) and not set(NEW) & set(SCENARIOS)

TRAITS = {
    "Vijay_Deewar": "Vijay (Deewar): a self-made man who grew up poor; proud, bitter about injustice, "
                    "speaks with quiet defiance and moral weight; never begs, never backs down.",
    "Don": "Don: a cool, calculating boss; confident, faintly amused, always a step ahead; "
           "short, controlled lines with a hint of threat.",
    "Shahenshah": "Shahenshah: a larger-than-life champion of justice with a royal manner; theatrical and lofty, "
                  "speaks in grand, Urdu-tinged words about honour, truth and dignity.",
    "Anthony_Gonsalves": "Anthony Gonsalves: a cheerful, comic Bombay boy; playful and warm, exaggerates for laughs, "
                         "mixes English freely, full of energy and jokes.",
    "Agneepath_Vijay": "Vijay Dinanath Chauhan (Agneepath): intense and terse; controlled anger, "
                       "short blunt sentences, driven by a long-held resolve.",
    "Jai_Sholay": "Jai (Sholay): calm, laconic, dry humour; a loyal, practical friend "
                  "who says little and keeps things steady.",
}
assert set(TRAITS) == set(PERSONAS)


def make_input(persona, scenario):
    return f"[Persona: {persona}] Traits: {TRAITS[persona]}\nScenario: {scenario}{TAIL}"


assert all(ex["input"] == f"[Persona: {ex['persona']}] Scenario: {ex['scenario']}{TAIL}" for ex in DATA)
for ex in DATA:
    ex["input"] = make_input(ex["persona"], ex["scenario"])

TRAIN = [ex for ex in DATA if ex["scenario"] not in HELD_OUT]
HELD = [ex for ex in DATA if ex["scenario"] in HELD_OUT]
EVAL = [{"persona": p, "scenario": s, "kind": "held-out" if s in HELD_OUT else "new",
         "input": make_input(p, s), "reference": next((ex["target"] for ex in HELD
                                                       if ex["persona"] == p and ex["scenario"] == s), None)}
        for s in HELD_OUT + NEW for p in PERSONAS]

print(f"personas {PERSONAS}")
print(f"train {len(TRAIN)} examples ({len(SCENARIOS) - len(HELD_OUT)} scenarios) | "
      f"held out {len(HELD)} ({len(HELD_OUT)} scenarios) | evaluation inputs {len(EVAL)}")
print("example input:\n" + TRAIN[0]["input"])

personas ['Vijay_Deewar', 'Don', 'Shahenshah', 'Anthony_Gonsalves', 'Agneepath_Vijay', 'Jai_Sholay']
train 48 examples (8 scenarios) | held out 12 (2 scenarios) | evaluation inputs 48
example input:
[Persona: Vijay_Deewar] Traits: Vijay (Deewar): a self-made man who grew up poor; proud, bitter about injustice, speaks with quiet defiance and moral weight; never begs, never backs down.
Scenario: Abhee chai ka mood hai. Generate one original Roman-script Hinglish dialogue line with this persona's broad dramatic traits. Do not quote existing films.


## Cell 3 — Model, template, and the training tensors

Each example is one user turn (the dataset's input, unchanged) and one model turn (the target line), in Gemma's official chat template (checked against the template's own rendering). The labels are `-100` everywhere except the answer and its closing `<end_of_turn>`, so the model learns the line and where to stop, not the prompt.

In [21]:
# Cell 3 — Load Gemma, render examples with its chat template, mask the prompt out of the labels

from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(GEMMA, revision=REV, token=HF_TOKEN)
base = AutoModelForCausalLM.from_pretrained(GEMMA, revision=REV, dtype=DTYPE, token=HF_TOKEN,
                                            attn_implementation="eager").to(DEVICE)
eos = base.generation_config.eos_token_id
STOPS = sorted(set(eos if isinstance(eos, list) else [eos]))


def prompt_text(user):
    msgs = [{"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)


def encode(ex):
    """Prompt tokens + answer tokens, tokenized apart (exactly what generation sees); the text must match
    Gemma's own rendering of the two-turn conversation."""
    msgs = [{"role": "user", "content": ex["input"]}, {"role": "assistant", "content": ex["target"]}]
    answer = ex["target"] + "<end_of_turn>\n"
    assert tok.apply_chat_template(msgs, tokenize=False) == prompt_text(ex["input"]) + answer
    prompt = tok(prompt_text(ex["input"]), add_special_tokens=False)["input_ids"]
    reply = tok(answer, add_special_tokens=False)["input_ids"]
    return torch.tensor(prompt + reply), torch.tensor([-100] * len(prompt) + reply)


def collate(examples):
    pairs = [encode(ex) for ex in examples]
    T = max(len(ids) for ids, _ in pairs)
    ids = torch.full((len(pairs), T), tok.pad_token_id)
    labels = torch.full((len(pairs), T), -100)
    mask = torch.zeros((len(pairs), T), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)], labels[i, :len(y)], mask[i, :len(x)] = x, y, 1
    return {"input_ids": ids.to(DEVICE), "attention_mask": mask.to(DEVICE), "labels": labels.to(DEVICE)}


ids, labels = encode(TRAIN[0])
print("stop token IDs", STOPS)
print("--- one training example as the model sees it:")
print(tok.decode(ids))
print("--- the part that carries the loss:")
print(repr(tok.decode(ids[labels != -100])))

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

stop token IDs [1, 106]
--- one training example as the model sees it:
<bos><start_of_turn>user
[Persona: Vijay_Deewar] Traits: Vijay (Deewar): a self-made man who grew up poor; proud, bitter about injustice, speaks with quiet defiance and moral weight; never begs, never backs down.
Scenario: Abhee chai ka mood hai. Generate one original Roman-script Hinglish dialogue line with this persona's broad dramatic traits. Do not quote existing films.<end_of_turn>
<start_of_turn>model
Chai peene ka mann hai, toh chai piyenge. Har chhoti khushi ke liye kisi ki ijaazat nahi maangte hum.<end_of_turn>

--- the part that carries the loss:
'Chai peene ka mann hai, toh chai piyenge. Har chhoti khushi ke liye kisi ki ijaazat nahi maangte hum.<end_of_turn>\n'


## Cell 4 — LoRA and training

- LoRA rank 16, alpha 32, dropout 0.05, on `q/k/v/o` and `gate/up/down` projections; everything else frozen.
- AdamW, learning rate 1e-4 with linear decay, batch 4, 3 epochs (36 steps). Run 1 used 2e-4 for 6 epochs and overfit after epoch 2.
- After every epoch: the mean training loss and the loss on the 12 held-out examples. Epoch 0 is the untouched model (LoRA starts as a no-op). The adapter from the epoch with the lowest held-out loss is kept.

In [22]:
# Cell 4 — Wrap with LoRA, train, keep the epoch with the lowest held-out loss

from peft import LoraConfig, get_peft_model

EPOCHS, BATCH, LR = 3, 4, 1e-4
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

torch.manual_seed(SEED)
model = get_peft_model(base, LORA)
model.print_trainable_parameters()

trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
steps = EPOCHS * -(-len(TRAIN) // BATCH)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / steps)


@torch.no_grad()
def held_out_loss():
    model.eval()
    return sum(model(**collate(HELD[i:i + BATCH])).loss.item() * len(HELD[i:i + BATCH])
               for i in range(0, len(HELD), BATCH)) / len(HELD)


def lora_state():
    return {k: v.detach().clone() for k, v in model.state_dict().items() if "lora_" in k}


LOG = [{"epoch": 0, "train_loss": None, "held_out_loss": round(held_out_loss(), 4)}]
best = (LOG[0]["held_out_loss"], 0, lora_state())
rng = random.Random(SEED)
t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = rng.sample(TRAIN, len(TRAIN))
    losses = []
    for i in range(0, len(order), BATCH):
        loss = model(**collate(order[i:i + BATCH])).loss
        loss.backward()
        opt.step()
        sched.step()
        opt.zero_grad()
        losses.append(loss.item())
    LOG.append({"epoch": epoch, "train_loss": round(sum(losses) / len(losses), 4),
                "held_out_loss": round(held_out_loss(), 4)})
    if LOG[-1]["held_out_loss"] < best[0]:
        best = (LOG[-1]["held_out_loss"], epoch, lora_state())
    print(LOG[-1], f"| {time.perf_counter() - t0:.0f}s")

model.load_state_dict(best[2], strict=False)
BEST_EPOCH = best[1]
print(f"kept epoch {BEST_EPOCH} (held-out loss {best[0]}); epoch 0 = untouched Gemma {LOG[0]['held_out_loss']}")

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879
{'epoch': 1, 'train_loss': 5.1757, 'held_out_loss': 4.0378} | 11s
{'epoch': 2, 'train_loss': 4.0903, 'held_out_loss': 3.8785} | 23s
{'epoch': 3, 'train_loss': 3.7215, 'held_out_loss': 3.8578} | 35s
kept epoch 3 (held-out loss 3.8578); epoch 0 = untouched Gemma 6.3648


## Cell 5 — Generate: untouched Gemma and Gemma + LoRA on the same 48 inputs

Two held-out and six new scenarios, six personas each. The untouched model is the same object with the adapter switched off (`disable_adapter`), so the only difference is the LoRA weights. Decoding is greedy with one guard against loops: no word 3-gram may appear twice (`no_repeat_ngram_size=3`), for both variants. Each output records how it ended (stop token or token limit) and its Devanagari share.

In [23]:
# Cell 5 — Greedy generation for both variants

DEVA = re.compile(r"[ऀ-ॿ]")
LATN = re.compile(r"[A-Za-z]")
NO_REPEAT = 3


@torch.no_grad()
def generate(user):
    ids = tok(prompt_text(user), return_tensors="pt", add_special_tokens=False).to(DEVICE)
    out = model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS,
                         no_repeat_ngram_size=NO_REPEAT, pad_token_id=tok.pad_token_id,
                         temperature=None, top_p=None, top_k=None)
    new = out[0, ids["input_ids"].shape[1]:].tolist()
    text = tok.decode(new, skip_special_tokens=True).strip()
    d, l = len(DEVA.findall(text)), len(LATN.findall(text))
    return {"output": text, "new_tokens": len(new),
            "ended": "stop token" if new and new[-1] in STOPS else "token limit",
            "devanagari_share": round(d / (d + l), 3) if d + l else None}


model.eval()
RESULTS = []
for variant in ["gemma (untouched)", "gemma + lora"]:
    t0 = time.perf_counter()
    for e in EVAL:
        if variant == "gemma (untouched)":
            with model.disable_adapter():
                r = generate(e["input"])
        else:
            r = generate(e["input"])
        RESULTS.append({"variant": variant, **{k: e[k] for k in ("kind", "scenario", "persona", "reference")}, **r})
    print(f"{variant}: {len(EVAL)} outputs in {time.perf_counter() - t0:.0f}s")

gemma (untouched): 48 outputs in 183s
gemma + lora: 48 outputs in 96s


## Cell 6 — Content quality: every output, side by side

For each scenario and persona: the untouched model, then the LoRA model, then the reference line where one exists (held-out scenarios only). These are what the human rubric scores. A short summary of the automatic signals follows: they are diagnostic only, not quality scores.

In [24]:
# Cell 6 — Outputs per scenario and persona, then the automatic signals per variant

VARIANTS = list(dict.fromkeys(r["variant"] for r in RESULTS))


def get(variant, scenario, persona):
    return next(r for r in RESULTS if (r["variant"], r["scenario"], r["persona"]) == (variant, scenario, persona))


for s in HELD_OUT + NEW:
    print(f"\n=== [{'held-out' if s in HELD_OUT else 'new'}] {s}")
    for p in PERSONAS:
        print(f"  {p}")
        for v in VARIANTS:
            r = get(v, s, p)
            print(f"    {v:18s} | {r['output']}" + ("" if r["ended"] == "stop token" else "   [TOKEN LIMIT]"))
        if s in HELD_OUT:
            print(f"    {'reference':18s} | {get(VARIANTS[0], s, p)['reference']}")

def repeats_sentence(t):
    sentences = [s.strip().lower() for s in re.split(r"[.?!]", t) if len(s.strip()) > 12]
    return len(sentences) != len(set(sentences))


print("\nAUTOMATIC SIGNALS (diagnostic only)")
for v in VARIANTS:
    rs = [r for r in RESULTS if r["variant"] == v]
    n = len(rs)
    print(f"  {v:18s} | stop token {sum(r['ended'] == 'stop token' for r in rs)}/{n}"
          f" | repeats a sentence {sum(repeats_sentence(r['output']) for r in rs)}/{n}"
          f" | opens with 'Okay' {sum(r['output'].startswith('Okay') for r in rs)}/{n}"
          f" | any Devanagari {sum((r['devanagari_share'] or 0) > 0 for r in rs)}/{n}"
          f" | mostly Devanagari {sum((r['devanagari_share'] or 0) > 0.5 for r in rs)}/{n}"
          f" | median tokens {sorted(r['new_tokens'] for r in rs)[n // 2]}")


=== [held-out] Koi tumhe gusse mein provoke kar raha hai.
  Vijay_Deewar
    gemma (untouched)  | Okay, here’s a dialogue line, aiming to capture Vijay Deewar’s essence, in a Roman-Script Hinglish style, with a bit of dramatic weight and a subtle undercurrent of bitterness:

**Dialogue:**

“Listen, you. You’re attempting to stir a pot of discontent with your… *gestures dismissively* …’arguments.’  You think a   [TOKEN LIMIT]
    gemma + lora       | Gupta nahi. Sabnein, sabnein. Humne apni-apni baat karna, apni- apni sahi baat karna.
    reference          | Tumhari awaaz unchi hai, iska matlab yeh nahi ki tum sahi ho. Main jawab dunga, lekin apne tareeke se.
  Don
    gemma (untouched)  | Okay, here’s a dialogue line in Roman script, fitting the persona you described, aiming for a slightly unsettling, controlled tone:

“Observe, my dear. A slight tremor in your composure. It’s… remarkably predictable. Like a moth drawn to a flickering flame.”

---

Let me know if you’d like me to try

## Cell 7 — Persona differentiation (automatic)

For each scenario: the mean word overlap (Jaccard) between the six personas' lines, taken over all 15 pairs, and the number of identical lines. Lower overlap means the personas say different things. The same number on the 10 training scenarios' reference lines gives the yardstick for what the dataset itself calls *different*.

A second check for copying: the share of each output's word 4-grams that also appear in some training line. High values on new scenarios mean the model is reciting training lines instead of answering.

In [25]:
# Cell 7 — Overlap between personas per scenario, and 4-gram copying from the training lines

def words(t):
    return re.findall(r"[a-zऀ-ॿ]+", t.lower())


def jaccard(a, b):
    a, b = set(words(a)), set(words(b))
    return len(a & b) / len(a | b) if a | b else 1.0


def mean_overlap(lines):
    return sum(jaccard(a, b) for a, b in combinations(lines, 2)) / 15


def grams(t, n=4):
    w = words(t)
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}


TRAIN_GRAMS = set().union(*(grams(ex["target"]) for ex in TRAIN))


def copied(t):
    g = grams(t)
    return len(g & TRAIN_GRAMS) / len(g) if g else 0.0


ref = [mean_overlap([ex["target"] for ex in DATA if ex["scenario"] == s]) for s in SCENARIOS]
print(f"yardstick: dataset reference lines, mean overlap between personas {sum(ref) / len(ref):.3f} "
      f"(range {min(ref):.3f}-{max(ref):.3f})\n")

DIFF = {}
print(f"{'scenario':58s} " + " | ".join(f"{v:>18s}" for v in VARIANTS))
for s in HELD_OUT + NEW:
    row = []
    for v in VARIANTS:
        lines = [get(v, s, p)["output"] for p in PERSONAS]
        DIFF[(v, s)] = {"overlap": round(mean_overlap(lines), 3), "identical": 6 - len(set(lines))}
        row.append(f"{DIFF[(v, s)]['overlap']:.3f} ({DIFF[(v, s)]['identical']} same)")
    print(f"{s[:58]:58s} " + " | ".join(f"{x:>18s}" for x in row))

for v in VARIANTS:
    d = [DIFF[(v, s)]["overlap"] for s in HELD_OUT + NEW]
    c = [copied(r["output"]) for r in RESULTS if r["variant"] == v and r["kind"] == "new"]
    print(f"\n{v:18s} | mean overlap {sum(d) / len(d):.3f} | identical lines {sum(DIFF[(v, s)]['identical'] for s in HELD_OUT + NEW)}"
          f" | new scenarios: mean 4-gram copying {sum(c) / len(c):.2f}, outputs > 50% copied {sum(x > 0.5 for x in c)}/{len(c)}")

lora = [r for r in RESULTS if r["variant"] == "gemma + lora"]
overlap = sum(DIFF[("gemma + lora", s)]["overlap"] for s in HELD_OUT + NEW) / len(HELD_OUT + NEW)
repeats = sum(repeats_sentence(r["output"]) for r in lora)
print("\nSUCCESS BAR (automatic part; readability and blind matching are human checks)")
print(f"  overlap between personas {overlap:.3f} (pass <= 0.15, run 1 0.303): {'PASS' if overlap <= 0.15 else 'FAIL'}")
print(f"  outputs repeating a sentence {repeats}/{len(lora)} (pass <= 4, run 1 16): {'PASS' if repeats <= 4 else 'FAIL'}")

yardstick: dataset reference lines, mean overlap between personas 0.087 (range 0.065-0.130)

scenario                                                    gemma (untouched) |       gemma + lora
Koi tumhe gusse mein provoke kar raha hai.                     0.169 (0 same) |     0.125 (0 same)
Aaj tumhara pehla din hai ek naye shehar mein.                 0.177 (0 same) |     0.208 (0 same)
Boss ne Friday shaam ko achaanak weekend ka kaam de diya.      0.173 (0 same) |     0.250 (0 same)
Tumhara purana dost saalon baad airport par mil gaya.          0.227 (0 same) |     0.389 (0 same)
Presentation se paanch minute pehle office ka WiFi band ho     0.178 (0 same) |     0.217 (0 same)
Baarish mein gaadi beech sadak par band ho gayi.               0.180 (0 same) |     0.163 (0 same)
Team ka sabse senior saathi aaj retire ho raha hai.            0.186 (0 same) |     0.132 (0 same)
Kisi ne tumse kaha ki tum yeh kaam kabhi nahi kar paoge.       0.197 (0 same) |     0.169 (0 same)

gemma (untouche

## Cell 8 — Blind persona matching (for a human reviewer)

The real differentiation test. For each scenario, the LoRA model's six lines are shuffled and labelled A–F. A reviewer who knows the six characters matches each letter to a persona without looking at Cell 9. If the matches are mostly right, the personas are distinct and recognisable; if they are near chance (1 in 6), they are not.

In [26]:
# Cell 8 — Shuffled LoRA outputs per scenario, letters only (the key is in Cell 9)

KEY = {}
shuffle = random.Random(SEED + 1)
print("Personas: Vijay (Deewar), Don, Shahenshah, Anthony Gonsalves, Vijay Dinanath Chauhan (Agneepath), Jai (Sholay)")
for s in HELD_OUT + NEW:
    order = shuffle.sample(PERSONAS, len(PERSONAS))
    KEY[s] = dict(zip("ABCDEF", order))
    print(f"\n=== {s}")
    for letter, p in KEY[s].items():
        print(f"  {letter}. {get('gemma + lora', s, p)['output']}")

Personas: Vijay (Deewar), Don, Shahenshah, Anthony Gonsalves, Vijay Dinanath Chauhan (Agneepath), Jai (Sholay)

=== Koi tumhe gusse mein provoke kar raha hai.
  A. Gupta nahi. Sabnein, sabnein. Humne apni-apni baat karna, apni- apni sahi baat karna.
  B. Gusse ka naam kya hai? Kaise usse sabaz nahi hai?
  C. Koi nahi, hum khud ko nahi. Humse khud ko chupne ke liye, khud ko khud nahi.
  D. Chai ka sab kuch nahi hai. Agar tumse koi guss hai, toh usse sab kuch kaise guss karo.
  E. Gusse nahi. Mujhe humein sab kuch nahi hai.
  F. Gusse ko baant kar, khud ko baanti nahi.

=== Aaj tumhara pehla din hai ek naye shehar mein.
  A. Chai ka chai, aaj tumhara din hai. Sab kuch hai hai, aur sab kuch hai tumhara. Humari khushi hai, humari khushiyon hai.
  B. Duniya ko dekh ke hum sab kuch nahi samajh sakte. lekin yeh shehar, yeh ek new start hai. Humne yeh start kiya, aur yeh start phir bhi happens.
  C. Din hai pehle hai. Sab kuch khushiyon se pehle. Sab kitni khushyon se pehli din hai.
  D. Dost,

## Cell 9 — The key, and every record

The answer key for Cell 8, then the training log, settings, and every result as JSON lines between markers, so the scoring sheets can be rebuilt from the saved notebook alone.

In [27]:
# Cell 9 — Key for the blind sheet, settings, and the full records

print("KEY")
for s, k in KEY.items():
    print(f"  {s[:50]:50s} " + "  ".join(f"{letter}={p}" for letter, p in k.items()))

SETTINGS = {
    "versions": VERSIONS,
    "data": {"url": DATA_URL, "train": len(TRAIN), "held_out": len(HELD), "held_out_scenarios": HELD_OUT,
             "new_scenarios": NEW},
    "lora": {"r": LORA.r, "alpha": LORA.lora_alpha, "dropout": LORA.lora_dropout,
             "targets": sorted(LORA.target_modules)},
    "training": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "schedule": "linear decay", "seed": SEED,
                 "kept_epoch": BEST_EPOCH, "log": LOG},
    "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "stop_token_ids": STOPS,
                   "no_repeat_ngram_size": NO_REPEAT,
                   "template": "Gemma official chat template, dataset input + traits line as the user turn, no system turn"},
    "traits": TRAITS,
}
print("\n----- SETTINGS -----")
print(json.dumps(SETTINGS, ensure_ascii=False, indent=1))
print("\n----- RECORDS (JSON lines) -----")
for r in RESULTS:
    print(json.dumps(r, ensure_ascii=False))
print("----- END RECORDS -----")

KEY
  Koi tumhe gusse mein provoke kar raha hai.         A=Vijay_Deewar  B=Shahenshah  C=Don  D=Anthony_Gonsalves  E=Agneepath_Vijay  F=Jai_Sholay
  Aaj tumhara pehla din hai ek naye shehar mein.     A=Anthony_Gonsalves  B=Agneepath_Vijay  C=Jai_Sholay  D=Shahenshah  E=Vijay_Deewar  F=Don
  Boss ne Friday shaam ko achaanak weekend ka kaam d A=Agneepath_Vijay  B=Shahenshah  C=Anthony_Gonsalves  D=Don  E=Vijay_Deewar  F=Jai_Sholay
  Tumhara purana dost saalon baad airport par mil ga A=Vijay_Deewar  B=Jai_Sholay  C=Anthony_Gonsalves  D=Agneepath_Vijay  E=Shahenshah  F=Don
  Presentation se paanch minute pehle office ka WiFi A=Agneepath_Vijay  B=Anthony_Gonsalves  C=Don  D=Jai_Sholay  E=Shahenshah  F=Vijay_Deewar
  Baarish mein gaadi beech sadak par band ho gayi.   A=Anthony_Gonsalves  B=Vijay_Deewar  C=Agneepath_Vijay  D=Shahenshah  E=Jai_Sholay  F=Don
  Team ka sabse senior saathi aaj retire ho raha hai A=Vijay_Deewar  B=Don  C=Shahenshah  D=Agneepath_Vijay  E=Jai_Sholay  F=Anthony_Gonsa

## Cell 10 — Optional: save the adapter

Writes the kept adapter (a few MB) to `/kaggle/working/persona-pilot-adapter`. Files in `/kaggle/working` are lost when an interactive session ends, so download it if the pilot is worth keeping. Nothing is uploaded anywhere.

In [28]:
# Cell 10 — Save the kept LoRA adapter locally on Kaggle

model.save_pretrained("/kaggle/working/persona-pilot-adapter")
!ls -la /kaggle/working/persona-pilot-adapter

total 51028
drwxr-xr-x 2 root root     4096 Oct 18 17:41 .
drwxr-xr-x 4 root root     4096 Oct 18 17:24 ..
-rw-r--r-- 1 root root     1176 Oct 18 17:41 adapter_config.json
-rw------- 1 root root 52231312 Oct 18 17:41 adapter_model.safetensors
-rw-r--r-- 1 root root     5194 Oct 18 17:41 README.md


---
## Next

Save the notebook (clear the token in Cell 0 first). Then:

1. **Content quality:** are most LoRA lines readable and on-topic?
2. **Differentiation:** a reviewer fills in the blind sheet (Cell 8) before looking at the key.
3. **Decision against the success bar:**
   - all four pass: expand toward 360 examples, with the traits line kept in the input;
   - personas separate but the Hinglish stays broken: Gemma 1B's Hinglish is the limit, and more data will not fix it;
   - neither improves: stop and rethink before collecting more data.